---
## 🚀 NOVELTY: Learning Functional Species Roles via GNN + Clustering

Instead of hand-crafted metrics, we:
1. Build the bipartite pollination graph
2. Train a Graph Autoencoder (GAE) using GraphSAGE convolutions to learn **node embeddings**
3. Cluster embeddings with K-Means → identify **ecological roles**
4. Compare GNN-discovered roles vs classical metric-based roles

This pipeline captures multi-hop structural patterns that degree/centrality metrics miss.

### Step 1 — Install dependencies (run once)

In [3]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
!pip install torch_geometric

Looking in indexes: https://download.pytorch.org/whl/cpu
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for torchvision from https://download-r2.pytorch.org/whl/cpu/torchvision-0.26.0%2Bcpu-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for networkx>=2.5.1 from https://files.pythonhosted.org/packages/b9/54/dd730b32ea14ea797530a4479b2ed46a6fb250f682a9cfb997e968bf0261/networkx-3.4.2-py3-none-any.whl.metadata
  Obtaining dependency information for fsspec>=0.8.5 from https://files.pythonhosted.org/packages/e6/ab/fb21f4c939bb440104cc2b396d3be1d9b7a9fd3c6c2a53d98c45b3d7c954/fsspec-2026.2.0-py3-none-any.whl.metadata
  Obtaining dependency information for jinja2 from https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata
Discarding https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl (from https://download.pytorch.org

     ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
      --------------------------------------- 0.0/1.3 MB 1.3 MB/s eta 0:00:01
     -- ------------------------------------- 0.1/1.3 MB 1.3 MB/s eta 0:00:01
     ------- -------------------------------- 0.2/1.3 MB 1.7 MB/s eta 0:00:01
     --------------- ------------------------ 0.5/1.3 MB 2.7 MB/s eta 0:00:01
     ------------------------------- -------- 1.0/1.3 MB 4.5 MB/s eta 0:00:01
     ---------------------------------------  1.3/1.3 MB 4.7 MB/s eta 0:00:01
     ---------------------------------------- 1.3/1.3 MB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/78.4 kB ? eta -:--:--
     ---------------------------------------- 78.4/78.4 kB 2.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/122.8 kB ? eta -:--:--
     -------------------------------------- 122.8/122.8 kB 3.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/462.0 kB ? eta -:--:--
     


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\akash\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [4]:
# Uncomment and run once if not already installed
!pip install torch torch_geometric scikit-learn matplotlib seaborn

     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/8.9 MB 1.3 MB/s eta 0:00:07
     ---------------------------------------- 0.1/8.9 MB 1.3 MB/s eta 0:00:07
      --------------------------------------- 0.2/8.9 MB 1.6 MB/s eta 0:00:06
     - -------------------------------------- 0.3/8.9 MB 1.8 MB/s eta 0:00:05
     -- ------------------------------------- 0.5/8.9 MB 2.2 MB/s eta 0:00:04
     -- ------------------------------------- 0.6/8.9 MB 2.5 MB/s eta 0:00:04
     --- ------------------------------------ 0.8/8.9 MB 2.6 MB/s eta 0:00:04
     ---- ----------------------------------- 1.0/8.9 MB 2.9 MB/s eta 0:00:03
     ----- ---------------------------------- 1.3/8.9 MB 3.1 MB/s eta 0:00:03
     ------ --------------------------------- 1.5/8.9 MB 3.4 MB/s eta 0:00:03
     ------- -------------------------------- 1.7/8.9 MB 3.5 MB/s eta 0:00:03
     -------- ------------------------------- 2.0/8.9 MB 3.6 MB/s eta 0


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\akash\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


### Step 2 — Imports & reproducibility

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import random, os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── PyTorch + PyG ──
import torch
torch.manual_seed(SEED)
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import from_networkx, to_undirected
from torch_geometric.nn import SAGEConv

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Reuse variables from earlier cells ──
# G_orig, df, pollinators, plants, metrics_df, OUT_DIR  (already defined above)
# If running this section standalone, load your data first.

Device: cpu


### Step 3 — Build PyG Data object with rich node features

In [8]:
def build_pyg_data(G: nx.Graph) -> tuple:
    """
    Annotate each node with structural features and convert to a
    PyTorch Geometric Data object.

    Features per node (6-dim):
        0  degree (raw)
        1  weighted degree
        2  betweenness centrality
        3  clustering coefficient
        4  bipartite side  (0=plant, 1=pollinator)
        5  log(degree + 1)  - scale-invariant degree signal
    """
    btw = nx.betweenness_centrality(G, weight="weight", normalized=True)
    clust = nx.clustering(G)

    node_order = list(G.nodes())
    rows = []
    for n in node_order:
        deg = float(G.degree(n))
        wdeg = float(sum(d.get("weight", 1.0) for _, _, d in G.edges(n, data=True)))
        side = float(G.nodes[n].get("bipartite", 0))
        rows.append([
            deg,
            wdeg,
            btw.get(n, 0.0),
            clust.get(n, 0.0),
            side,
            float(np.log1p(deg)),
        ])

    X_raw = np.array(rows, dtype=np.float32)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw).astype(np.float32)

    pyg_data = from_networkx(G)
    pyg_data.x = torch.tensor(X_scaled, dtype=torch.float)
    pyg_data.node_names = node_order

    print(f"PyG Data: {pyg_data}")
    print(f"Feature matrix: {pyg_data.x.shape}  (nodes x features)")
    return pyg_data, node_order, X_raw


def load_df_for_gnn() -> pd.DataFrame:
    """Load dataframe when running novelty section standalone."""
    if "df" in globals() and isinstance(df, pd.DataFrame):
        return df

    candidate_paths = []
    if "DATA_PATH" in globals() and isinstance(DATA_PATH, str):
        candidate_paths.append(DATA_PATH)

    candidate_paths.extend([
        r"C:\Users\akash\Videos\Mtech_IIT\2nd sem\network science\project\ns\data\M_PL_001.csv",
        os.path.join("data", "M_PL_001.csv"),
    ])

    for path in candidate_paths:
        if isinstance(path, str) and os.path.exists(path):
            print(f"Loading dataframe from: {path}")
            loaded_df = pd.read_csv(path, index_col=0).fillna(0).astype(float)
            globals()["df"] = loaded_df
            globals()["DATA_PATH"] = path
            return loaded_df

    raise NameError(
        "No graph/dataframe available. Define DATA_PATH to a valid CSV, or load df/G_orig before this cell."
    )


def resolve_graph_for_gnn() -> nx.Graph:
    """Find an existing graph or build one from df for standalone runs."""
    if "G_orig" in globals() and isinstance(G_orig, nx.Graph):
        print("Using existing graph: G_orig")
        return G_orig

    if "graphs" in globals() and isinstance(graphs, list) and len(graphs) > 0:
        if isinstance(graphs[0], nx.Graph):
            print("Using first graph from graphs list: graphs[0]")
            return graphs[0]

    local_df = load_df_for_gnn()
    print("Building graph from dataframe...")
    G = nx.Graph()
    plant_nodes = list(local_df.index)
    poll_nodes = list(local_df.columns)
    G.add_nodes_from(plant_nodes, bipartite=0, kind="plant")
    G.add_nodes_from(poll_nodes, bipartite=1, kind="pollinator")

    for p in plant_nodes:
        for q in poll_nodes:
            w = float(local_df.loc[p, q])
            if w > 0:
                G.add_edge(p, q, weight=w)

    globals()["plants"] = plant_nodes
    globals()["pollinators"] = poll_nodes
    globals()["G_orig"] = G
    return G


G_for_gnn = resolve_graph_for_gnn()
pyg_data, node_order, X_raw_feats = build_pyg_data(G_for_gnn)
pyg_data = pyg_data.to(DEVICE)

Loading dataframe from: C:\Users\akash\Videos\Mtech_IIT\2nd sem\network science\project\ns\data\M_PL_001.csv
Building graph from dataframe...
PyG Data: Data(edge_index=[2, 722], bipartite=[185], kind=[185], weight=[722], num_nodes=185, x=[185, 6], node_names=[185])
Feature matrix: torch.Size([185, 6])  (nodes x features)


### Step 4 — Define GraphSAGE Autoencoder

In [9]:
class GraphSAGEEncoder(nn.Module):
    """
    Two-layer GraphSAGE encoder.
    Input  → hidden_dim → embedding_dim
    """
    def __init__(self, in_channels: int, hidden_dim: int, embedding_dim: int,
                 dropout: float = 0.3):
        super().__init__()
        self.conv1   = SAGEConv(in_channels, hidden_dim)
        self.conv2   = SAGEConv(hidden_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        self.bn1     = nn.BatchNorm1d(hidden_dim)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.elu(h)
        h = self.dropout(h)
        h = self.conv2(h, edge_index)
        return h   # shape: [N, embedding_dim]


class GraphAutoencoder(nn.Module):
    """
    Graph Autoencoder:
      Encoder  : GraphSAGE  (structural embedding)
      Decoder  : inner-product  (reconstruct adjacency)
      Aux head : feature reconstruction  (MLP)
    """
    def __init__(self, in_channels: int, hidden_dim: int = 64,
                 embedding_dim: int = 32, dropout: float = 0.3):
        super().__init__()
        self.encoder = GraphSAGEEncoder(in_channels, hidden_dim,
                                        embedding_dim, dropout)
        # Auxiliary decoder: reconstruct input features
        self.feat_decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, in_channels),
        )

    def encode(self, x, edge_index):
        return self.encoder(x, edge_index)

    def decode_adj(self, z):
        """Inner-product decoder → reconstructed adjacency logits."""
        return torch.sigmoid(z @ z.t())

    def decode_feats(self, z):
        return self.feat_decoder(z)

    def forward(self, x, edge_index):
        z = self.encode(x, edge_index)
        return z, self.decode_adj(z), self.decode_feats(z)


IN_DIM   = pyg_data.x.shape[1]   # 6 features
HID_DIM  = 64
EMBD_DIM = 32

model = GraphAutoencoder(IN_DIM, HID_DIM, EMBD_DIM).to(DEVICE)
print(model)
print(f"\nTrainable parameters: {sum(p.numel() for p in model.parameters()):,}")

GraphAutoencoder(
  (encoder): GraphSAGEEncoder(
    (conv1): SAGEConv(6, 64, aggr=mean)
    (conv2): SAGEConv(64, 32, aggr=mean)
    (dropout): Dropout(p=0.3, inplace=False)
    (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (feat_decoder): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=6, bias=True)
  )
)

Trainable parameters: 7,590


### Step 5 — Training loop

In [10]:
def build_adj_target(edge_index: torch.Tensor, n_nodes: int) -> torch.Tensor:
    """Construct dense adjacency matrix from edge_index (for loss computation)."""
    A = torch.zeros(n_nodes, n_nodes)
    src, dst = edge_index
    A[src, dst] = 1.0
    A[dst, src] = 1.0   # undirected
    return A.to(DEVICE)


N = pyg_data.num_nodes
A_target = build_adj_target(pyg_data.edge_index, N)
# Positive-class weight to handle sparsity
pos_weight = torch.tensor([(N * N - A_target.sum()) / A_target.sum()]).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300)

EPOCHS     = 300
ALPHA_FEAT = 0.5   # weight for feature reconstruction loss

loss_history = []

model.train()
for epoch in range(1, EPOCHS + 1):
    optimizer.zero_grad()

    z, A_recon, X_recon = model(pyg_data.x, pyg_data.edge_index)

    # 1) Adjacency reconstruction loss (weighted BCE)
    loss_adj = F.binary_cross_entropy(
        A_recon, A_target,
        weight=(A_target * (pos_weight - 1) + 1)
    )

    # 2) Feature reconstruction loss
    loss_feat = F.mse_loss(X_recon, pyg_data.x)

    loss = loss_adj + ALPHA_FEAT * loss_feat
    loss.backward()
    optimizer.step()
    scheduler.step()

    loss_history.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS}  "
              f"loss={loss.item():.4f}  "
              f"adj={loss_adj.item():.4f}  "
              f"feat={loss_feat.item():.4f}")

print("\nTraining complete ✓")

Epoch  50/300  loss=1.3609  adj=1.2614  feat=0.1990
Epoch 100/300  loss=1.2504  adj=1.1609  feat=0.1789
Epoch 150/300  loss=1.1419  adj=1.0920  feat=0.0997
Epoch 200/300  loss=1.1117  adj=1.0709  feat=0.0815
Epoch 250/300  loss=1.1034  adj=1.0696  feat=0.0676
Epoch 300/300  loss=1.1053  adj=1.0715  feat=0.0677

Training complete ✓


### Step 6 — Extract embeddings & cluster into ecological roles

In [11]:
model.eval()
with torch.no_grad():
    Z = model.encode(pyg_data.x, pyg_data.edge_index).cpu().numpy()

print(f"Embedding matrix shape: {Z.shape}  (185 species × {EMBD_DIM} dims)")

# ── Optimal k via silhouette score ──
sil_scores = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    lbl = km.fit_predict(Z)
    sil_scores[k] = silhouette_score(Z, lbl)
    print(f"  k={k}  silhouette={sil_scores[k]:.4f}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\n→ Best k = {best_k} (silhouette={sil_scores[best_k]:.4f})")

# ── Final clustering with k=4 (matches ecological role taxonomy) ──
K_ROLES = 4
km_final = KMeans(n_clusters=K_ROLES, random_state=SEED, n_init=50)
cluster_labels = km_final.fit_predict(Z)

# ── Assign semantic role names using raw structural features ──
# For each cluster, compute mean raw degree and betweenness
X_raw = np.array(X_raw_feats)   # shape [N, 6]
centers_raw = np.array([
    X_raw[cluster_labels == k].mean(axis=0) for k in range(K_ROLES)
])  # [K, 6]  columns: deg, wdeg, btw, clust, side, log_deg

deg_col = 0
btw_col = 2
med_deg = np.median(centers_raw[:, deg_col])
med_btw = np.median(centers_raw[:, btw_col])

ROLE_NAMES = []
for k in range(K_ROLES):
    d = centers_raw[k, deg_col]
    b = centers_raw[k, btw_col]
    if d > med_deg and b > med_btw:
        ROLE_NAMES.append("Keystone")
    elif d > med_deg:
        ROLE_NAMES.append("Generalist")
    elif b > med_btw:
        ROLE_NAMES.append("Connector")
    else:
        ROLE_NAMES.append("Specialist")

# Guarantee unique names (fallback: priority assignment by degree)
if len(set(ROLE_NAMES)) < K_ROLES:
    order_by_deg = np.argsort(centers_raw[:, deg_col])[::-1]
    priority = ["Generalist", "Keystone", "Connector", "Specialist"]
    ROLE_NAMES = [""] * K_ROLES
    for rank, k in enumerate(order_by_deg):
        ROLE_NAMES[k] = priority[rank]

gnn_role_series = pd.Series(
    {node_order[i]: ROLE_NAMES[cluster_labels[i]] for i in range(len(node_order))},
    name="gnn_role"
)

print("\nGNN-discovered role distribution (all species):")
print(gnn_role_series.value_counts())

# Pollinator-only subset
poll_set = set(pollinators)
gnn_poll_roles = gnn_role_series[gnn_role_series.index.isin(poll_set)]
print("\nGNN roles — pollinators only:")
print(gnn_poll_roles.value_counts())

Embedding matrix shape: (185, 32)  (185 species × 32 dims)
  k=2  silhouette=0.2882
  k=3  silhouette=0.2508
  k=4  silhouette=0.2806
  k=5  silhouette=0.2844
  k=6  silhouette=0.2832
  k=7  silhouette=0.2515

→ Best k = 2 (silhouette=0.2882)

GNN-discovered role distribution (all species):
gnn_role
Specialist    72
Keystone      67
Connector     41
Generalist     5
Name: count, dtype: int64

GNN roles — pollinators only:
gnn_role
Specialist    57
Keystone      30
Connector     11
Generalist     3
Name: count, dtype: int64


### Step 7 — Comparison: Classical roles vs GNN roles

In [13]:
def build_classical_metrics_if_needed() -> pd.DataFrame:
    """Build a fallback metrics_df for pollinators when missing."""
    if "metrics_df" in globals() and isinstance(metrics_df, pd.DataFrame):
        return metrics_df

    if "G_for_gnn" in globals() and isinstance(G_for_gnn, nx.Graph):
        G_ref = G_for_gnn
    elif "G_orig" in globals() and isinstance(G_orig, nx.Graph):
        G_ref = G_orig
    else:
        raise NameError("metrics_df is missing and no graph is available to rebuild it.")

    deg = nx.degree_centrality(G_ref)
    btw = nx.betweenness_centrality(G_ref, weight="weight", normalized=True)
    clust = nx.clustering(G_ref)

    poll_nodes = [
        n for n, d in G_ref.nodes(data=True)
        if d.get("kind") == "pollinator" or d.get("bipartite") == 1
    ]
    if not poll_nodes:
        poll_nodes = list(G_ref.nodes())

    rows = []
    for n in poll_nodes:
        rows.append({
            "species": n,
            "degree": G_ref.degree(n),
            "degree_centrality": deg.get(n, 0.0),
            "betweenness_centrality": btw.get(n, 0.0),
            "clustering_coefficient": clust.get(n, 0.0),
        })

    mdf = pd.DataFrame(rows).set_index("species")

    # Classical 4-role assignment using degree and betweenness quartiles
    q_deg_low, q_deg_high = mdf["degree"].quantile([0.25, 0.75])
    q_btw_low, q_btw_high = mdf["betweenness_centrality"].quantile([0.25, 0.75])

    def assign_role(r):
        if r["degree"] >= q_deg_high and r["betweenness_centrality"] >= q_btw_high:
            return "Keystone"
        if r["degree"] >= q_deg_high:
            return "Generalist"
        if r["betweenness_centrality"] >= q_btw_high:
            return "Connector"
        return "Specialist"

    mdf["role"] = mdf.apply(assign_role, axis=1)
    globals()["metrics_df"] = mdf
    print("metrics_df was missing; rebuilt a fallback version from graph centralities.")
    return mdf


metrics_df = build_classical_metrics_if_needed()

# Align indices: only pollinators appear in metrics_df
common_idx = metrics_df.index.intersection(gnn_poll_roles.index)
classical_roles = metrics_df.loc[common_idx, "role"]
gnn_roles_aligned = gnn_poll_roles.loc[common_idx]

# Map roles to integers for ARI
all_role_labels = ["Generalist", "Specialist", "Connector", "Keystone"]
role2int = {r: i for i, r in enumerate(all_role_labels)}
y_classical = classical_roles.map(role2int).values
y_gnn = gnn_roles_aligned.map(role2int).values

ari = adjusted_rand_score(y_classical, y_gnn)
print(f"Adjusted Rand Index (classical vs GNN): {ari:.4f}")
print("(ARI=1.0 -> perfect agreement; ARI~0 -> random; negative -> worse than random)")

# Confusion-style agreement table
agreement_df = pd.crosstab(
    classical_roles.rename("Classical"),
    gnn_roles_aligned.rename("GNN"),
    margins=True
)
print("\nRole agreement table:")
print(agreement_df)

metrics_df was missing; rebuilt a fallback version from graph centralities.
Adjusted Rand Index (classical vs GNN): 0.3710
(ARI=1.0 -> perfect agreement; ARI~0 -> random; negative -> worse than random)

Role agreement table:
GNN         Connector  Generalist  Keystone  Specialist  All
Classical                                                   
Connector           0           0         0           4    4
Generalist          0           0         5           0    5
Keystone            0           3        18           1   22
Specialist         11           0         7          52   70
All                11           3        30          57  101


### Step 8 — Visualisation: GNN embedding space + role assignments

In [15]:
BG   = "#0D1117"
TXT  = "#E6EDF3"
GRID = "#30363D"
ROLE_COLORS = {
    "Generalist" : "#4CAF50",
    "Specialist"  : "#42A5F5",
    "Connector"   : "#FF7043",
    "Keystone"    : "#FFD700",
}

def ensure_out_dir() -> str:
    if "OUT_DIR" in globals() and isinstance(OUT_DIR, str) and OUT_DIR.strip():
        out_dir = OUT_DIR
    else:
        out_dir = os.path.join(os.getcwd(), "outputs")
        globals()["OUT_DIR"] = out_dir
    os.makedirs(out_dir, exist_ok=True)
    return out_dir

OUT_DIR = ensure_out_dir()

def style_ax(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, color=TXT, fontsize=10, fontweight="bold", pad=8)
    ax.tick_params(colors=TXT)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)
    ax.xaxis.label.set_color(TXT)
    ax.yaxis.label.set_color(TXT)

# -- PCA of embedding for 2D scatter --
pca2 = PCA(n_components=2, random_state=SEED)
Z2   = pca2.fit_transform(Z)
var_exp = pca2.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle(
    "GNN-Learned Functional Species Roles - Pollination Network M_PL_001",
    color=TXT, fontsize=13, fontweight="bold", y=1.02
)

# -- Panel A: PCA of GNN embeddings, coloured by GNN role --
ax = axes[0]
style_ax(ax, "A) GNN Embedding Space (PCA)")
for i, node in enumerate(node_order):
    role   = gnn_role_series.get(node, "Specialist")
    color  = ROLE_COLORS[role]
    kind   = G_orig.nodes[node].get("kind", "pollinator")
    marker = "s" if kind == "plant" else "o"
    ax.scatter(Z2[i, 0], Z2[i, 1],
               c=color, marker=marker, s=30, alpha=0.75, linewidths=0)

ax.set_xlabel(f"PC1 ({var_exp[0]:.1f}% var)")
ax.set_ylabel(f"PC2 ({var_exp[1]:.1f}% var)")
legend_els = [
    mpatches.Patch(color=c, label=r) for r, c in ROLE_COLORS.items()
] + [
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor="grey",
               markersize=7, label="plant"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="grey",
               markersize=7, label="pollinator"),
]
ax.legend(handles=legend_els, fontsize=7, facecolor=BG, labelcolor=TXT, loc="best")

# -- Panel B: Silhouette score vs k --
ax2 = axes[1]
style_ax(ax2, "B) Silhouette Score vs Number of Clusters")
ks   = list(sil_scores.keys())
vals = list(sil_scores.values())
ax2.plot(ks, vals, "-o", color="#42A5F5", lw=2, markersize=7)
ax2.axvline(K_ROLES, color="#FFD700", ls="--", lw=1.5, label=f"k={K_ROLES} chosen")
ax2.set_xlabel("k (number of clusters)")
ax2.set_ylabel("Silhouette Score")
ax2.legend(fontsize=8, facecolor=BG, labelcolor=TXT)
ax2.grid(color=GRID, ls="--", lw=0.5)

# -- Panel C: Stacked bar - Classical vs GNN role distribution --
ax3 = axes[2]
style_ax(ax3, "C) Classical vs GNN Role Distribution (pollinators)")

def role_counts(series):
    vc = series.value_counts()
    return {r: vc.get(r, 0) for r in all_role_labels}

classic_counts = role_counts(classical_roles)
gnn_counts     = role_counts(gnn_poll_roles.loc[common_idx])

x      = np.arange(2)
labels = ["Classical\n(KMeans+metrics)", "GNN\n(GraphSAGE+KMeans)"]
bottom = np.zeros(2)
for role in all_role_labels:
    vals_bar = [classic_counts[role], gnn_counts[role]]
    ax3.bar(x, vals_bar, bottom=bottom, label=role,
            color=ROLE_COLORS[role], alpha=0.88)
    for xi, (v, b) in enumerate(zip(vals_bar, bottom)):
        if v > 0:
            ax3.text(xi, b + v / 2, str(v), ha="center", va="center",
                     color="white", fontsize=9, fontweight="bold")
    bottom += np.array(vals_bar)

ax3.set_xticks(x)
ax3.set_xticklabels(labels, color=TXT)
ax3.set_ylabel("Number of pollinators")
ax3.legend(fontsize=8, facecolor=BG, labelcolor=TXT, loc="upper right")
ax3.text(0.5, -0.12, f"ARI = {ari:.3f}",
         transform=ax3.transAxes, ha="center",
         color="#FF7043", fontsize=10, fontweight="bold")
ax3.grid(axis="y", color=GRID, ls="--", lw=0.5)

plt.tight_layout()
gnn_fig_path = os.path.join(OUT_DIR, "gnn_species_roles.png")
fig.savefig(gnn_fig_path, dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"\nSaved figure -> {gnn_fig_path}")


Saved figure -> c:\Users\akash\Downloads\outputs\gnn_species_roles.png


### Step 9 — Training loss curve

In [16]:
if "OUT_DIR" not in globals() or not isinstance(OUT_DIR, str) or not OUT_DIR.strip():
    OUT_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

fig_loss, ax_loss = plt.subplots(figsize=(8, 4), facecolor=BG)
style_ax(ax_loss, "GNN Autoencoder - Training Loss Curve")
ax_loss.plot(loss_history, color="#42A5F5", lw=1.5)
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Total Loss")
ax_loss.grid(color=GRID, ls="--", lw=0.5)
plt.tight_layout()
loss_path = os.path.join(OUT_DIR, "gnn_training_loss.png")
fig_loss.savefig(loss_path, dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print(f"Saved loss curve -> {loss_path}")

Saved loss curve -> c:\Users\akash\Downloads\outputs\gnn_training_loss.png


### Step 10 — Export GNN roles to CSV & print summary

In [17]:
if "OUT_DIR" not in globals() or not isinstance(OUT_DIR, str) or not OUT_DIR.strip():
    OUT_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# Merge GNN roles into metrics_df
gnn_combined = pd.DataFrame({
    "species"    : node_order,
    "kind"       : [G_orig.nodes[n].get("kind", "?") for n in node_order],
    "gnn_cluster": cluster_labels,
    "gnn_role"   : [gnn_role_series[n] for n in node_order],
    "pca1"       : Z2[:, 0],
    "pca2"       : Z2[:, 1],
}).set_index("species")

# Add classical role for pollinators
gnn_combined["classical_role"] = gnn_combined.index.map(
    lambda s: metrics_df.loc[s, "role"] if s in metrics_df.index else np.nan
)

csv_path = os.path.join(OUT_DIR, "gnn_species_roles.csv")
gnn_combined.to_csv(csv_path)
print(f"Saved GNN roles -> {csv_path}")

print("\n" + "=" * 58)
print("GNN NOVELTY - SUMMARY")
print("=" * 58)
print(f"  GNN architecture          GraphSAGE Autoencoder")
print(f"  Input features / node     {IN_DIM}  (degree, wdeg, btw, clust, side, log-deg)")
print(f"  Hidden dim / Embed dim    {HID_DIM} / {EMBD_DIM}")
print(f"  Training epochs           {EPOCHS}")
print(f"  Final loss                {loss_history[-1]:.4f}")
print(f"  Silhouette (k=4)          {sil_scores[4]:.4f}")
print(f"  Best k (silhouette)       {best_k}")
print(f"  ARI vs classical roles    {ari:.4f}")
print("-" * 58)
print("GNN-discovered roles (all species):")
for r, c in gnn_role_series.value_counts().items():
    print(f"    {r:<14} {c}")
print("=" * 58)

Saved GNN roles -> c:\Users\akash\Downloads\outputs\gnn_species_roles.csv

GNN NOVELTY - SUMMARY
  GNN architecture          GraphSAGE Autoencoder
  Input features / node     6  (degree, wdeg, btw, clust, side, log-deg)
  Hidden dim / Embed dim    64 / 32
  Training epochs           300
  Final loss                1.1053
  Silhouette (k=4)          0.2806
  Best k (silhouette)       2
  ARI vs classical roles    0.3710
----------------------------------------------------------
GNN-discovered roles (all species):
    Specialist     72
    Keystone       67
    Connector      41
    Generalist     5
